#0. Imports & Configs

In [0]:
import gc
import os
import sys
import time
from datetime import date, datetime
from getpass import getuser

# Third-party Data Science & ML
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from sklearn.linear_model import LinearRegression
from sklearn.metrics._scorer import _SCORERS

# PySpark
from pyspark import SparkConf, SparkContext
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import PandasUDFType, col, lit, pandas_udf, to_date
from pyspark.sql.types import (DateType, DoubleType, FloatType, StringType, StructField, StructType)

In [0]:
ORIGIN_TABLE = "samples.bakehouse.sales_transactions"
LINEAR_REGRESSION_TABLE = "workspace.forecasting.linear_regression_forecast"
TYPE_PERIOD="day"
RANGE_PREDICTION = 5

#1. Input Building

In [0]:
# This is a use of the bakehouse sample of Databricks
# It's important that the input follows this structure (dt_period for the date, cd_key for the subject to be predict and qt_units for the quantity)

query = f"""
SELECT
TRY_CAST(dateTime AS DATE) dt_period,
CONCAT(product, ';', paymentMethod) cd_key,
SUM(quantity) qt_units
FROM
{ORIGIN_TABLE}
GROUP BY ALL
"""

In [0]:
df = spark.sql(query)

output_schema = StructType([
    StructField("cd_key", StringType(), True),
    StructField("dt_period", DateType(), True),
    StructField("qt_units", DoubleType(), True),
    StructField("fl_type", StringType(), True)
])

#2. Forecasting (Linear Regression)

In [0]:
def linear_regression(pdf: pd.DataFrame) -> pd.DataFrame:

    arg_name = f"{TYPE_PERIOD}s"

    # --- PREPARATION AND TRAINING ---
    pdf = pdf.sort_values('dt_period')
    pdf['dt_period'] = pd.to_datetime(pdf['dt_period'])
    
    ## Ordinal conversion
    X = pdf['dt_period'].map(datetime.toordinal).values.reshape(-1, 1)
    y = pdf['qt_units']
    
    model = LinearRegression()
    model.fit(X, y)
    
    # --- PREDICTION ---
    last_date = pdf['dt_period'].max()
    future_dates = [last_date + relativedelta(**{arg_name: i}) for i in range(1, RANGE_PREDICTION + 1)]
    
    X_future = np.array([d.toordinal() for d in future_dates]).reshape(-1, 1)
    y_pred = model.predict(X_future)
    y_pred = np.maximum(0, y_pred)
    
    key_atual = pdf['cd_key'].iloc[0]
    
    # --- DATAFRAMES BUILDING ---
    
    ## Prediction's DataFrame
    df_linear_regression = pd.DataFrame({
        'cd_key': key_atual,
        'dt_period': future_dates,
        'qt_units': y_pred.astype(int),
        'fl_type': 'Linear Regression'
    })
    
    ## Record DataFrame
    df_record = pdf[['cd_key', 'dt_period', 'qt_units']].copy()
    df_record['fl_type'] = 'Record'
    
    # --- CONCAT ---
    df_final = pd.concat([df_record, df_linear_regression], ignore_index=True)
    
    return df_final

In [0]:
df_forecast = df.groupBy("cd_key").applyInPandas(linear_regression, schema=output_schema)

#3. Table Writing

In [0]:
spark.sql(f"""
  CREATE CATALOG IF NOT EXISTS {LINEAR_REGRESSION_TABLE.split('.')[0]}
""")
spark.sql(f"""
  CREATE DATABASE IF NOT EXISTS {LINEAR_REGRESSION_TABLE.split('.')[0]}.{LINEAR_REGRESSION_TABLE.split('.')[1]}
""")
df_forecast.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(LINEAR_REGRESSION_TABLE)